# Experiment 2 – External economic shock
Reads the BehaviorSpace export of the shock experiment (spreadsheet **or** table format) and reproduces the before/after, composition, recovery and dose-response analyses. Time axis is the tick count (`[step]` in a spreadsheet export, `ticks` in a table export).

**Optional paired control.** The Experiment 1 sweep uses the same seeds and the same code up to the shock tick, so its runs are the no-shock counterfactual for every run here. Set `CONTROL_FILE` to that table export to get paired shock effects (Section 5); set it to `None` to skip.

In [ ]:
import csv, numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
plt.style.use('ggplot')

FILE         = 'External_Shock_2.csv'        # <- Experiment 2 BehaviorSpace export (table, every 10 ticks)
CONTROL_FILE = 'Diaspora_sweep_v2_8.csv'     # <- Experiment 1 table export (no shock, same seeds) or None
SHOCK_TICK   = 793                           # last tick BEFORE the shock (go commands: go / if ticks = 794 [ apply-economic-shock ])
NET, DIA, SEED = 'network-strength', 'diaspora-signal', 'global-seed'
COLORS = ['cyan', 'green', 'yellow', 'red', 'orange']
LABEL  = {'cyan': 'cyan (voluntary non-migrant)', 'green': 'green (voluntary migrant)', 'yellow': 'yellow (acquiescent)',
          'red': 'red (trapped)', 'orange': 'orange (distressed migrant)'}
PLOT   = {'cyan': 'darkturquoise', 'green': 'forestgreen', 'yellow': 'gold', 'red': 'crimson', 'orange': 'darkorange'}

In [ ]:
def load_export(path):
    """Read a BehaviorSpace export, spreadsheet or table format, into one long DataFrame (one row per run and tick)."""
    rows = list(csv.reader(open(path, encoding='utf-8-sig')))
    if 'Spreadsheet' in ''.join(rows[0]):
        i_run = next(i for i, r in enumerate(rows) if r and r[0] == '[run number]')
        i_all = next(i for i, r in enumerate(rows) if r and r[0] == '[all run data]')
        hdr = rows[i_all][1:]
        starts = [j for j, h in enumerate(hdr) if h == '[step]']
        L = starts[1] - starts[0] if len(starts) > 1 else len(hdr)
        seen, names = {}, []
        for h in hdr[:L]:                               # a metric listed twice gets a '.1' suffix
            seen[h] = seen.get(h, -1) + 1
            names.append(h if seen[h] == 0 else f'{h}.{seen[h]}')
        params = {rows[j][0]: j for j in range(i_run + 1, i_all) if rows[j] and rows[j][0] and not rows[j][0].startswith('[')}
        body = np.array([r[1:] for r in rows[i_all + 1:] if len(r) > 1], dtype=object)
        out = []
        for k in range(len(hdr) // L):
            c0 = k * L
            b = pd.DataFrame(body[:, c0:c0 + L], columns=names).apply(pd.to_numeric, errors='coerce').dropna(subset=['[step]'])
            b['[run number]'] = int(rows[i_run][1 + c0])
            for p, j in params.items():
                b[p] = rows[j][1 + c0]
            out.append(b)
        d = pd.concat(out, ignore_index=True)
        for p in params:
            d[p] = d[p].map(_num)
        d = d.rename(columns={'[step]': 'ticks'})
    else:                                               # table export: 6 metadata lines, then the header
        d = pd.read_csv(path, skiprows=6)
        d.columns = [c.strip() for c in d.columns]
        if 'ticks' not in d.columns:
            d = d.rename(columns={'[step]': 'ticks'})
    dup = [c for c in d.columns if c.endswith('.1') and c[:-2] in d.columns]
    d = d.drop(columns=dup)                             # repeated metric columns
    d = d.rename(columns={c: c.split('=')[-1].strip(' ]') for c in d.columns if c.startswith('count turtles with [color')})
    d = d.rename(columns={c: 'fin-ability-orange' for c in d.columns if c.startswith('ifelse-value any? turtles with [color = orange]')})
    d = d.rename(columns={c: 'mean-asp-economic' for c in d.columns if c.strip() == 'mean [asp-economic] of turtles'})
    return d

def _num(v):
    try: return float(v)
    except (TypeError, ValueError): return v

d = load_export(FILE)
runs = d['[run number]'].nunique()
print(f"{len(d):,} rows | {runs} runs | rows/run {d.groupby('[run number]').size().unique()} | ticks {d.ticks.min():.0f}-{d.ticks.max():.0f}")
print('runs per arm:'); print(d.groupby([NET, DIA]).size().unstack().div(d.ticks.nunique()).rename_axis(index='network', columns='diaspora').astype(int))
assert d[SEED].nunique() * d[NET].nunique() * d[DIA].nunique() == runs, 'design is not fully crossed'
T_END = d.ticks.max()

# the shock must show up as one big drop in mean ability between two consecutive recorded ticks
rec_ticks = np.sort(d.ticks.unique())
T_PRE  = rec_ticks[rec_ticks <= SHOCK_TICK].max()       # last recorded tick before the shock
T_POST = rec_ticks[rec_ticks >  SHOCK_TICK].min()       # first recorded tick after it
drop_tick = d.groupby('ticks')['mean-ability-level'].mean().diff().idxmin()
print(f'recorded every {np.diff(rec_ticks).min():.0f} tick(s); largest drop in mean ability at tick {drop_tick:.0f}; snapshots: before = {T_PRE:.0f}, after = {T_POST:.0f}')
assert drop_tick == T_POST, 'SHOCK_TICK does not match the data'

In [ ]:
d

## 1. What the shock does at once (tick before vs tick after)
Same runs, the last recorded tick before the shock and the first after it. With 10-tick recording the gap includes six ordinary ticks of growth, so the change is almost entirely the shock.

In [ ]:
snap_cols = ['mean-aspiration-level', 'mean-ability-level', 'share-high-ability', 'mean-asp-social', 'mobility-zone-share',
             'unique-migrants', 'cumulative-migrations', 'cumulative-exits'] + COLORS
snap_cols += [c for c in ('mean-asp-economic', 'fin-ability-orange') if c in d.columns]
a = d[d.ticks == T_PRE].set_index('[run number]')[snap_cols]
b = d[d.ticks == T_POST].set_index('[run number]')[snap_cols]
snap = pd.DataFrame({'before': a.mean(), 'after': b.mean()})
snap['change'] = snap.after - snap.before
snap['change_%'] = 100 * snap.change / snap.before
snap['p_paired_t'] = [stats.ttest_rel(b[c], a[c]).pvalue if (b[c] - a[c]).std() > 0 else np.nan for c in snap_cols]
snap.assign(p_paired_t=snap.p_paired_t.map('{:.1e}'.format)).round(3)

## 2. Time course across the whole run

In [ ]:
G = d.groupby('ticks')[['mean-aspiration-level', 'mean-ability-level', 'share-high-ability', 'mobility-zone-share',
                        'unique-migrants', 'cumulative-migrations', 'cumulative-exits'] + COLORS].mean()

fig, axes = plt.subplots(2, 2, figsize=(12, 7.5))
ax = axes[0, 0]
ax.plot(G.index, G['mean-aspiration-level'], label='mean total aspiration'); ax.plot(G.index, G['mean-ability-level'], label='mean total ability')
ax.axhline(5, color='grey', ls=':', lw=1); ax.set_title('Aspiration and ability'); ax.set_ylabel('level (0-10)'); ax.legend(loc='lower right')
ax = axes[0, 1]
ax.stackplot(G.index, *[G[c] for c in COLORS], colors=[PLOT[c] for c in COLORS], labels=[LABEL[c] for c in COLORS], alpha=.85)
ax.set_title('Agent composition (mean count of 200)', pad=30); ax.set_ylabel('agents'); ax.legend(loc='lower center', bbox_to_anchor=(.5, 1.0), ncol=3, fontsize=6.5, frameon=False); ax.set_ylim(0, 200)
ax = axes[1, 0]
ax.plot(G.index, G['unique-migrants'], label='first-time migrants'); ax.plot(G.index, G['cumulative-migrations'], label='cumulative entries')
ax.plot(G.index, G['cumulative-exits'], label='cumulative exits', ls='--'); ax.set_title('Entries and exits'); ax.set_ylabel('count'); ax.legend(loc='upper left')
ax = axes[1, 1]
ax.plot(G.index, G['mobility-zone-share'], color='C3', label='in zone now'); ax.plot(G.index, G['share-high-ability'], color='C4', label='share high ability')
ax.set_title('Mobility-zone share and high-ability share'); ax.set_ylabel('share of agents'); ax.legend(loc='lower right')
for ax in axes.ravel():
    ax.axvline(SHOCK_TICK, color='k', ls='--', lw=1); ax.set_xlabel('ticks')
axes[0, 0].text(SHOCK_TICK + 15, 5.1, 'shock', fontsize=8)
plt.tight_layout(); plt.show()

## 3. Who is moved by the shock? Composition before, after and at the end

In [ ]:
comp = G.loc[[T_PRE, T_POST, T_END], COLORS].T
comp.columns = ['before shock', 'just after', 'end of run']
comp['shock change'] = comp['just after'] - comp['before shock']
comp['recovered by end'] = comp['end of run'] - comp['just after']
comp.rename(index=LABEL).round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
x = np.arange(len(COLORS)); w = .27
for i, (col, lab) in enumerate(zip(['before shock', 'just after', 'end of run'], ['before shock', 'just after', 'end of run'])):
    ax.bar(x + (i - 1) * w, comp[col].values, w, label=lab, edgecolor='black', color=['#bbbbbb', '#777777', '#333333'][i])
ax.set_xticks(x); ax.set_xticklabels(COLORS); ax.set_ylabel('mean agents'); ax.set_title('Composition by agent colour (all runs)'); ax.legend()
plt.tight_layout(); plt.show()

## 4. Recovery after the shock
How much of the one-tick loss in ability and aspiration has been regained by the end of the run? Ability can regain only through human-capital growth and savings; in these runs the shock-raised aspiration does not decay.

In [ ]:
rec = []
for m in ['mean-ability-level', 'share-high-ability', 'mean-aspiration-level', 'mobility-zone-share', 'red', 'orange']:
    pre, post, end = G.loc[T_PRE, m], G.loc[T_POST, m], G.loc[T_END, m]
    rec.append(dict(metric=m, before=pre, just_after=post, end=end, shock_change=post - pre,
                    share_of_shock_undone_by_end=(end - post) / (pre - post) if pre != post else np.nan))
pd.DataFrame(rec).set_index('metric').round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
Z = d[(d.ticks >= SHOCK_TICK - 100)]
for ax, m, t in zip(axes, ['mean-ability-level', 'mobility-zone-share'], ['Mean ability', 'Mobility-zone share']):
    for n, c in zip(sorted(d[NET].unique()), ['C0', 'C1']):
        z = Z[Z[NET] == n].groupby('ticks')[m].agg(['mean', 'sem'])
        ax.plot(z.index, z['mean'], color=c, label=f'network-strength = {n:g}'); ax.fill_between(z.index, z['mean'] - 1.96 * z['sem'], z['mean'] + 1.96 * z['sem'], color=c, alpha=.15)
    ax.axvline(SHOCK_TICK, color='k', ls='--', lw=1); ax.set_title(t + ' around the shock'); ax.set_xlabel('ticks'); ax.legend(loc='lower right')
plt.tight_layout(); plt.show()

## 5. Paired shock effect against the no-shock control
Same seed, same arm, with and without the shock (control = Experiment 1 export). Positive means the shock raised the value.

In [ ]:
HAVE_CTRL = CONTROL_FILE is not None
if HAVE_CTRL:
    c = load_export(CONTROL_FILE)
    METR = ['unique-migrants', 'cumulative-migrations', 'cumulative-exits', 'mean-ability-level', 'mean-aspiration-level', 'mobility-zone-share'] + COLORS
    METR = [m for m in METR if m in c.columns]
    def wide(df, m): return df.pivot_table(index=[NET, DIA, SEED], columns='ticks', values=m)
    common = sorted(set(d.ticks) & set(c.ticks))
    EFF = {m: wide(d, m)[common] - wide(c, m).reindex(wide(d, m).index)[common] for m in METR}
    pre = [t for t in common if t <= SHOCK_TICK]
    worst = max(EFF[m][pre].abs().max().max() for m in ['unique-migrants', 'mean-ability-level'])
    print(f'control ticks in common: {len(common)} | max pre-shock difference (must be 0): {worst}')
    assert worst == 0, 'control runs differ from the shock runs before the shock: not the same code/seeds'
else:
    print('CONTROL_FILE is None: Sections 5-6 skipped')

In [ ]:
if HAVE_CTRL:
    CHECK = [t for t in (T_POST, 900, 1000, 1200, 1500, 2000, 3000, 4000, 5000, T_END) if t in common]
    def boot(x, n=5000):
        rng = np.random.default_rng(0); x = np.asarray(x)
        bs = x[rng.integers(0, len(x), (n, len(x)))].mean(1)
        return np.percentile(bs, [2.5, 97.5])
    rows = []
    for m in METR:
        for t in CHECK:
            x = EFF[m][t].values; lo, hi = boot(x)
            rows.append(dict(metric=m, tick=t, shock_effect=x.mean(), lo=lo, hi=hi))
    eff_tab = pd.DataFrame(rows)
    display(eff_tab.pivot(index='metric', columns='tick', values='shock_effect').loc[METR].round(2))

In [ ]:
if HAVE_CTRL:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, ms, ttl in zip(axes, [['unique-migrants', 'cumulative-migrations'], ['red', 'orange', 'cyan', 'green', 'yellow'], ['cumulative-exits']],
                           ['Entries: shock minus no-shock', 'Agents by colour: shock minus no-shock', 'Exits: shock minus no-shock']):
        for m in ms:
            e = EFF[m]; mu = e.mean(); se = e.std() / np.sqrt(len(e))
            ax.plot(mu.index, mu.values, label=m, color=PLOT.get(m)); ax.fill_between(mu.index, mu - 1.96 * se, mu + 1.96 * se, alpha=.15, color=PLOT.get(m))
        ax.axhline(0, color='k', lw=.8); ax.axvline(SHOCK_TICK, color='k', ls='--', lw=1); ax.set_title(ttl); ax.set_xlabel('ticks'); ax.legend(loc='best', fontsize=8)
    plt.tight_layout(); plt.show()

In [ ]:
if HAVE_CTRL:
    # when does the shock-induced extra entry turn into a deficit?
    mu = EFF['unique-migrants'].mean()
    after = mu[mu.index > SHOCK_TICK]
    print('extra first-time migrants right after the shock:', round(after.iloc[0], 2), '| peak:', round(after.max(), 2), 'at tick', after.idxmax())
    cross = after[after <= 0]
    print('first tick the shock effect on first-time migrants is <= 0:', cross.index[0] if len(cross) else 'never within the run')
    print('effect at last common tick:', round(mu.iloc[-1], 2), '| cumulative entries:', round(EFF['cumulative-migrations'].mean().iloc[-1], 2),
          '| exits:', round(EFF['cumulative-exits'].mean().iloc[-1], 2))

## 6. Shock effect by arm (diaspora signal × network strength)

In [ ]:
if HAVE_CTRL:
    t_last = max(common)
    rows = []
    for (n, dv), grp in EFF['unique-migrants'].groupby(level=[0, 1]):
        r = dict(net=n, dia=dv)
        for m in ['unique-migrants', 'cumulative-migrations', 'red', 'orange']:
            r[f'{m} @{CHECK[0]}'] = EFF[m].loc[grp.index, CHECK[0]].mean()
            r[f'{m} @{t_last}'] = EFF[m].loc[grp.index, t_last].mean()
        rows.append(r)
    display(pd.DataFrame(rows).round(2))

## 7. Does the diaspora pull still act after the shock? (paired by seed, against signal = 0)

In [ ]:
def wide_d(m): return d.pivot_table(index=[NET, DIA, SEED], columns='ticks', values=m)
U = wide_d('unique-migrants'); C = wide_d('cumulative-migrations')

def paired(P, n, dv, t, base=0.0):
    x = (P.loc[(n, dv), t] - P.loc[(n, base), t]).values
    rng = np.random.default_rng(0); bs = x[rng.integers(0, len(x), (5000, len(x)))].mean(1)
    return dict(net=n, dia=dv, tick=t, gain=x.mean(), lo=np.percentile(bs, 2.5), hi=np.percentile(bs, 97.5),
                p_t=stats.ttest_rel(P.loc[(n, dv), t], P.loc[(n, base), t]).pvalue)

dose = pd.DataFrame([paired(U, n, dv, t) for t in (T_PRE, T_END) for n in sorted(d[NET].unique()) for dv in sorted(d[DIA].unique()) if dv > 0])
dose.assign(p_t=dose.p_t.map('{:.1e}'.format)).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, t, ttl in zip(axes, (T_PRE, T_END), ('just before the shock', 'end of run')):
    for n, c in zip(sorted(d[NET].unique()), ['C0', 'C1']):
        s = dose[(dose.net == n) & (dose.tick == t)]
        ax.errorbar(s.dia, s.gain, yerr=[s.gain - s.lo, s.hi - s.gain], marker='o', capsize=3, color=c, label=f'network-strength = {n:g}')
    ax.set_title(f'Gain in first-time migrants at t={t:.0f} ({ttl})'); ax.set_xlabel('diaspora-signal'); ax.legend(loc='upper left')
axes[0].set_ylabel('gain vs signal = 0 (paired, 95% CI)')
plt.tight_layout(); plt.show()

In [ ]:
# Entries after the shock only: increase from the post-shock tick to the end, by arm
post = pd.DataFrame({'first-time migrants after shock': (U[T_END] - U[T_POST]), 'cumulative entries after shock': (C[T_END] - C[T_POST])})
post.groupby([NET, DIA]).mean().round(2)

## 8. Agent colours at the end of the run, by diaspora signal

In [ ]:
fin = d[d.ticks == T_END].groupby([NET, DIA])[COLORS + ['unique-migrants', 'mean-aspiration-level', 'mean-ability-level']].mean().round(2)
fin

In [ ]:
# change in colour counts relative to signal = 0 (same network setting), end of run
fm = d[d.ticks == T_END].groupby([NET, DIA])[COLORS].mean()
chg = fm.sub(fm.xs(0.0, level=DIA), level=NET)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, n in zip(axes, sorted(d[NET].unique())):
    for col in COLORS:
        ax.plot(chg.loc[n].index, chg.loc[n, col], 'o-', color=PLOT[col], label=col)
    ax.axhline(0, color='k', lw=.8); ax.set_title(f'network-strength = {n:g}'); ax.set_xlabel('diaspora-signal')
axes[0].set_ylabel('change in mean agents vs signal = 0'); axes[0].legend(loc='best')
plt.tight_layout(); plt.show()

## 9. Phase table for the write-up
Mean over all runs, and over the active-social runs only (diaspora-signal > 0 and network-strength > 0).

In [ ]:
pts = {'before shock': T_PRE, 'just after shock': T_POST, 'T + 400': T_POST + 400, 'T + 2000': T_POST + 2000, 'end of run': T_END}
act = d[(d[DIA] > 0) & (d[NET] > 0)]
cols_ = ['unique-migrants', 'cumulative-migrations', 'cumulative-exits', 'mean-aspiration-level', 'mean-ability-level', 'mobility-zone-share']
out = {}
for name, df_ in (('all runs', d), ('diaspora > 0 and network > 0', act)):
    out[name] = pd.DataFrame({k: df_[df_.ticks == t][cols_].mean() for k, t in pts.items()})
pd.concat(out, axis=1).round(2)

## 10. Economic aspiration and distressed migrants' financial ability
Needs the two extra reporters (`mean [asp-economic] of turtles` and the guarded mean of `abil-financial` over orange agents). The orange figure is a mean over whoever is orange *at that tick*, so the group changes when the shock pushes agents out of orange.

In [ ]:
if 'mean-asp-economic' in d.columns:
    X = d.groupby('ticks')[['mean-asp-economic', 'fin-ability-orange', 'orange', 'mean-ability-level']].mean()
    a = d[d.ticks == T_PRE].set_index('[run number]'); b = d[d.ticks == T_POST].set_index('[run number]')
    print(f"economic aspiration: {a['mean-asp-economic'].mean():.2f} -> {b['mean-asp-economic'].mean():.2f} "
          f"(+{(b['mean-asp-economic'] - a['mean-asp-economic']).mean():.2f}, {100 * (b['mean-asp-economic'] / a['mean-asp-economic'] - 1).mean():.1f}%)")
    print(f"mean total ability: {a['mean-ability-level'].mean():.2f} -> {b['mean-ability-level'].mean():.2f} ({100 * (b['mean-ability-level'].mean() / a['mean-ability-level'].mean() - 1):.1f}%)")
    print(f"financial ability of orange agents: {a['fin-ability-orange'].mean():.2f} -> {b['fin-ability-orange'].mean():.2f} | orange agents {a['orange'].mean():.1f} -> {b['orange'].mean():.1f}")
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(X.index, X['mean-asp-economic']); axes[0].set_title('Mean economic aspiration'); axes[0].set_ylabel('0-10')
    axes[1].plot(X.index, X['fin-ability-orange'], color=PLOT['orange'], label='financial ability of orange agents')
    ax2 = axes[1].twinx(); ax2.plot(X.index, X['orange'], color='grey', ls='--', label='number of orange agents'); ax2.grid(False)
    axes[1].set_title('Distressed migrants: ability and headcount'); axes[1].legend(loc='upper left'); ax2.legend(loc='upper right')
    for ax in axes: ax.axvline(SHOCK_TICK, color='k', ls='--', lw=1); ax.set_xlabel('ticks')
    plt.tight_layout(); plt.show()
else:
    print('economic-aspiration / orange-ability reporters not in this export')

## Notes on what the export does and does not contain
* The shock raises each agent's economic aspiration by 2 to 5 and lowers every ability component by 0.3 × (11 − financial ability); the export records only the population means, so the paper's earlier group-level figures (for example distressed migrants' financial ability before and after) need an extra reporter, such as `mean [abil-financial] of turtles with [color = orange]`.
* The shock is applied once, when `ticks = SHOCK_TICK + 1` (794), to every arm, so the arms differ only in `diaspora-signal` and `network-strength`.
* Without `CONTROL_FILE` there is no no-shock comparison inside this file; before/after changes then mix the shock with ordinary growth.